In [62]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os

In [63]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

model = ChatGoogleGenerativeAI(
    model = 'gemini-1.5-flash',
    api_key = api_key
)

In [64]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    bpb: float
    boundary_percent: float
    summary: str
    


In [65]:
def calc_sr(state: BatsmanState)-> BatsmanState:

    sr = (state['runs']/state['balls'])*100
    
    return {'sr': sr}

In [66]:
def calc_bpb(state:BatsmanState)-> BatsmanState:

    bpb = state['balls']/(state['fours']+state['sixes'])
    

    return {'bpb':bpb}

In [67]:
def calc_boundary_percent(state:BatsmanState)-> BatsmanState:
    boundary_percent = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100
    return {'boundary_percent': boundary_percent}

In [68]:
def calc_summary(state: BatsmanState):

    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary Percent - {state['boundary_percent']}"""


    return {'summary': summary}

In [69]:
graph = StateGraph(BatsmanState)

graph.add_node('calc_sr', calc_sr)
graph.add_node('calc_bpb', calc_bpb)
graph.add_node('calc_boundary_percent', calc_boundary_percent)
graph.add_node('calc_summary', calc_summary)

In [70]:
graph.add_edge(START, 'calc_sr')
graph.add_edge(START, 'calc_bpb')
graph.add_edge(START, 'calc_boundary_percent')
graph.add_edge('calc_sr', 'calc_summary')
graph.add_edge('calc_bpb', 'calc_summary')
graph.add_edge('calc_boundary_percent', 'calc_summary')
graph.add_edge('calc_summary', END)

workflow = graph.compile()

In [71]:
initial_state = {
    'runs' : 100,
    'balls' : 40,
    'fours': 5,
    'sixes':8,
}

prompt = f''

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 40,
 'fours': 5,
 'sixes': 8,
 'sr': 250.0,
 'bpb': 3.076923076923077,
 'boundary_percent': 68.0,
 'summary': '\nStrike Rate - 250.0 \n\nBalls per boundary - 3.076923076923077 \n\nBoundary Percent - 68.0'}